<a href="https://colab.research.google.com/github/umiSirya/covid19_global_analysis/blob/main/covid19_global_eda_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# importing necessary libraries
import pandas as pd
import numpy as np

In [8]:
#  load the data
confirmed_global = pd.read_csv('time_series_covid19_confirmed_global.csv')
deaths_global    = pd.read_csv('time_series_covid19_deaths_global.csv')

print("Confirmed Global shape:", confirmed_global.shape)
print("Deaths Global shape:", deaths_global.shape)

Confirmed Global shape: (289, 1147)
Deaths Global shape: (289, 1147)


# **EDA**

In [15]:
# checking the first 5 rows
print("First 5 rows (Confirmed) ")
print(confirmed_global.iloc[:5, :8])

# data types for confirmed global
print("Data types")
print(confirmed_global.dtypes[:6])

# Identify the date columns vs info columns
date_cols = [col for col in confirmed_global.columns if col[0].isdigit()]
info_cols = ['Province/State', 'Country/Region', 'Lat', 'Long']

print("Date range in dataset")
print("First date:", date_cols[0])
print("Last date:", date_cols[-1])
print("Total days tracked:", len(date_cols))

print("\n--- Number of countries/regions ---")
print(confirmed_global['Country/Region'].nunique(), "unique countries/regions")


First 5 rows (Confirmed) 
  Province/State Country/Region       Lat       Long  1/22/20  1/23/20  \
0            NaN    Afghanistan  33.93911  67.709953        0        0   
1            NaN        Albania  41.15330  20.168300        0        0   
2            NaN        Algeria  28.03390   1.659600        0        0   
3            NaN        Andorra  42.50630   1.521800        0        0   
4            NaN         Angola -11.20270  17.873900        0        0   

   1/24/20  1/25/20  
0        0        0  
1        0        0  
2        0        0  
3        0        0  
4        0        0  
Data types
Province/State     object
Country/Region     object
Lat               float64
Long              float64
1/22/20             int64
1/23/20             int64
dtype: object
Date range in dataset
First date: 1/22/20
Last date: 3/9/23
Total days tracked: 1143

--- Number of countries/regions ---
201 unique countries/regions


In [16]:
# missing values
print(confirmed_global[info_cols].isnull().sum())



Province/State    198
Country/Region      0
Lat                 2
Long                2
dtype: int64


In [18]:
# finding province/state is null for 198 of 289 rows

print("Countries that report by province (Province/State not null)")
has_province = confirmed_global[confirmed_global['Province/State'].notnull()]
print(has_province['Country/Region'].unique())

Countries that report by province (Province/State not null)
['Australia' 'Canada' 'China' 'Denmark' 'France' 'Netherlands'
 'New Zealand' 'United Kingdom']


In [19]:
# finding the  2 rows have missing Lat/Long
print("Rows with missing Lat/Long")
print(confirmed_global[confirmed_global['Lat'].isnull()][['Province/State', 'Country/Region']])
# These are 'Repatriated Travellers, Canada' and 'Unknown, China'
# Both are non-geographic — they will be dropped in cleaning

Rows with missing Lat/Long
            Province/State Country/Region
53  Repatriated Travellers         Canada
89                 Unknown          China


In [21]:
# finding the date columns have zero nulls and zero negatives
print("Nulls in date columns:", confirmed_global[date_cols].isnull().sum().sum())
print("Negative values in date columns:", (confirmed_global[date_cols] < 0).sum().sum())


Nulls in date columns: 0
Negative values in date columns: 0


In [23]:
#TOTALS & SCALE OF THE DATA

latest_date = date_cols[-1]

print("Total CONFIRMED cases globally (as of", latest_date, ")")
print(f"{confirmed_global[latest_date].sum():,}")

print("Total DEATHS globally (as of", latest_date, ")")
print(f"{deaths_global[latest_date].sum():,}")

print("Global Death Rate")
total_confirmed = confirmed_global[latest_date].sum()
total_deaths    = deaths_global[latest_date].sum()
print(f"{(total_deaths / total_confirmed * 100).round(2)}%")

Total CONFIRMED cases globally (as of 3/9/23 )
676,570,149
Total DEATHS globally (as of 3/9/23 )
6,881,802
Global Death Rate
1.02%


In [24]:
# top 10 countries by confirmed cases

# Sum by country (handles countries split by province)
confirmed_by_country = confirmed_global.groupby('Country/Region')[latest_date].sum()
print("Top 10 countries by confirmed cases")
print(confirmed_by_country.nlargest(10))

Top 10 countries by confirmed cases
Country/Region
US                103802702
India              44690738
France             39866718
Germany            38249060
Brazil             37076053
Japan              33320438
Korea, South       30615522
Italy              25603510
United Kingdom     24658705
Russia             22075858
Name: 3/9/23, dtype: int64


In [25]:
# top 10 countires by deaths

deaths_by_country = deaths_global.groupby('Country/Region')[latest_date].sum()
print("Top 10 countries by deaths")
print(deaths_by_country.nlargest(10))

Top 10 countries by deaths
Country/Region
US                1123836
Brazil             699276
India              530779
Russia             388478
Mexico             333188
United Kingdom     220721
Peru               219539
Italy              188322
Germany            168935
France             166176
Name: 3/9/23, dtype: int64


In [27]:
# global cases over time

milestone_dates = ['3/1/20', '6/1/20', '1/1/21', '6/1/21', '1/1/22', '6/1/22', '1/1/23', date_cols[-1]]

print("Global confirmed cases at milestone dates")
for d in milestone_dates:
    if d in confirmed_global.columns:
        total = confirmed_global[d].sum()
        print(f"  {d}: {total:,}")

Global confirmed cases at milestone dates
  3/1/20: 88,402
  6/1/20: 6,283,580
  1/1/21: 84,342,314
  6/1/21: 171,700,134
  1/1/22: 289,931,428
  6/1/22: 530,939,976
  1/1/23: 660,778,387
  3/9/23: 676,570,149


# **Cleaning**

In [28]:
# fixing Province/State NULLS

confirmed_global['Province/State'] = confirmed_global['Province/State'].fillna(confirmed_global['Country/Region'])
deaths_global['Province/State']    = deaths_global['Province/State'].fillna(deaths_global['Country/Region'])


In [30]:

# melt the date columns into rows

confirmed_melted = confirmed_global.melt(
    id_vars=['Province/State', 'Country/Region', 'Lat', 'Long'],
    var_name='Date', value_name='Confirmed'
)
confirmed_melted['Date'] = pd.to_datetime(confirmed_melted['Date'], format='mixed')

deaths_melted = deaths_global.melt(
    id_vars=['Province/State', 'Country/Region', 'Lat', 'Long'],
    var_name='Date', value_name='Deaths'
)
deaths_melted['Date'] = pd.to_datetime(deaths_melted['Date'], format='mixed')

# merge confirmed and deaths

global_df = pd.merge(
    confirmed_melted,
    deaths_melted[['Province/State', 'Country/Region', 'Date', 'Deaths']],
    on=['Province/State', 'Country/Region', 'Date'],
    how='left'
)

# groupby country and date

global_df = global_df.groupby(['Country/Region', 'Date'], as_index=False).agg(
    Confirmed=('Confirmed', 'sum'),
    Deaths=('Deaths', 'sum')
)

In [31]:

# adding a calculated column:Death Rate
global_df['Death_Rate_%'] = np.where(
    global_df['Confirmed'] > 0,
    (global_df['Deaths'] / global_df['Confirmed'] * 100).round(2),
    0.0
)

# Time columns
global_df['Year']       = global_df['Date'].dt.year
global_df['Month']      = global_df['Date'].dt.month
global_df['Month_Name'] = global_df['Date'].dt.strftime('%b')



In [32]:
# checking on data after changes
print("Shape:", global_df.shape)
print("\nMissing values:\n", global_df.isnull().sum())
print("\nDate range:", global_df['Date'].min(), "to", global_df['Date'].max())
print("Countries:", global_df['Country/Region'].nunique())



Shape: (229743, 8)

Missing values:
 Country/Region    0
Date              0
Confirmed         0
Deaths            0
Death_Rate_%      0
Year              0
Month             0
Month_Name        0
dtype: int64

Date range: 2020-01-22 00:00:00 to 2023-03-09 00:00:00
Countries: 201


In [33]:

# export data to csv

global_df.to_csv('global_covid_clean.csv', index=False)
print("\n✅ Exported: global_covid_clean.csv")


✅ Exported: global_covid_clean.csv
